# 01 · Data Collection: Market Stress Early-Warning System

**Research question (working):** Can signals from market networks, financial text sentiment, anomaly detection, and time-series forecasting warn of U.S. equity market stress earlier than standard financial stress indexes do?

This notebook is the first pass at the data pipeline. It:

1. Pulls benchmark stress indexes and market series from **FRED**
2. Pulls daily prices for **SPY and the nine original sector ETFs** from Yahoo Finance
3. Aligns everything on one trading-day calendar and computes log returns
4. Builds a **preliminary stress label** from the St. Louis Fed Financial Stress Index
5. Computes two first-look network-style features from the literature: rolling **average pairwise correlation** and the **absorption ratio** (Kritzman et al., 2011)
6. Saves a clean panel to `data/processed/` for the anomaly, forecasting, and network notebooks

The loading code lives in `src/data_loading.py` so the other notebooks can reuse it.

In [ ]:
# Run once per environment (or: pip install -r requirements.txt)
# %pip install pandas numpy requests python-dotenv yfinance matplotlib pyarrow

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from src import data_loading as dl

pd.set_option("display.width", 120)
START = "1999-01-01"

## 1. FRED series

If you have a free FRED API key, put it in a `.env` file in the repo root as `FRED_API_KEY=...`. Without a key the loader uses FRED's public CSV download, which works too.

| Series | What it is | Why we want it |
|---|---|---|
| `STLFSI4` | St. Louis Fed Financial Stress Index (weekly) | Candidate benchmark / label (context Q2) |
| `NFCI` | Chicago Fed National Financial Conditions Index (weekly) | Second benchmark to compare against |
| `VIXCLS` | CBOE VIX (daily) | Market-implied equity volatility |
| `BAMLH0A0HYM2` | High-yield corporate bond spread (daily) | Credit stress |
| `T10Y2Y` | 10y minus 2y Treasury spread (daily) | Yield-curve signal |

In [ ]:
fred = dl.load_fred(start=START)
print(fred.shape)
fred.tail()

In [ ]:
# Coverage check: date range, observation count, and native frequency per series
pd.DataFrame({
    "first": fred.apply(lambda s: s.first_valid_index()),
    "last": fred.apply(lambda s: s.last_valid_index()),
    "n_obs": fred.notna().sum(),
    # Typical spacing between observations: ~1 day for daily series, 7 for weekly
    "median_gap_days": fred.apply(lambda s: s.dropna().index.to_series().diff().dt.days.median()),
})

## 2. ETF prices

SPY plus the nine original Select Sector SPDRs (XLB, XLE, XLF, XLI, XLK, XLP, XLU, XLV, XLY). XLRE and XLC are left out for now because they start in 2015 and 2018; adding them would shorten the usable history.

In [ ]:
prices = dl.load_prices(start=START)
print(prices.shape)
prices.tail()

In [ ]:
prices.isna().sum()

## 3. Align on trading days and add returns

Weekly indexes are forward-filled onto trading days. **Caution:** this uses the observation date, not the release date. Before we train any forecasting model, weekly series need to be shifted by their publication lag or we will leak future information (context Q6).

In [ ]:
panel = dl.build_daily_panel(fred, prices)
print(panel.shape, panel.index.min().date(), "to", panel.index.max().date())
panel.filter(like="_ret").describe().T.round(4)

## 4. Preliminary stress label

`STLFSI4` is built so that 0 is roughly normal conditions and positive values are above-average stress. As a **placeholder**, a day is labeled stressed when `STLFSI4 > 1`. The group still needs to decide the definition and benchmark (context Q1 to Q3).

In [ ]:
panel["stress"] = dl.stress_flag(panel["STLFSI4"], threshold=1.0)
print(f"Share of trading days flagged as stressed: {panel['stress'].mean():.1%}")

# Contiguous stressed spells (start, end, length) for a quick look
spell_id = (panel["stress"].diff() != 0).cumsum()
spells = (panel[panel["stress"] == 1]
          .groupby(spell_id[panel["stress"] == 1])
          .apply(lambda g: pd.Series({"start": g.index.min().date(),
                                      "end": g.index.max().date(),
                                      "trading_days": len(g)})))
if spells.empty:
    print("No stressed days at this threshold.")
else:
    display(spells.sort_values("trading_days", ascending=False).head(10))

In [ ]:
def shade_stress(ax, flag):
    on = flag.astype(bool)
    starts = on.index[on & ~on.shift(1, fill_value=False)]
    ends = on.index[on & ~on.shift(-1, fill_value=False)]
    for s, e in zip(starts, ends):
        ax.axvspan(s, e, color="tab:red", alpha=0.15, lw=0)

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
panel["STLFSI4"].plot(ax=axes[0], color="black", lw=1)
axes[0].axhline(1.0, ls="--", color="tab:red", lw=1)
axes[0].set_title("St. Louis Fed Financial Stress Index (shaded = preliminary stress label)")
panel["VIXCLS"].plot(ax=axes[1], color="tab:blue", lw=1); axes[1].set_title("VIX")
(np.log(panel["SPY"])).plot(ax=axes[2], color="tab:green", lw=1); axes[2].set_title("log SPY price")
for ax in axes:
    shade_stress(ax, panel["stress"])
plt.tight_layout(); plt.show()

## 5. First-look network features

Two simple measures of how tightly the sector ETFs move together:

* **Average pairwise correlation** over a 60-day window
* **Absorption ratio** (Kritzman, Li, Page & Rigobon, 2011): share of total variance explained by the top principal components over a 250-day window. This is a simplified version (plain rolling covariance instead of exponential weighting).

These feed directly into context question 4: do they rise *before* stress, or only during it?

In [ ]:
sector_rets = panel[[f"{t}_ret" for t in dl.SECTOR_ETFS]]
panel["avg_corr_60d"] = dl.rolling_avg_correlation(sector_rets, window=60)
panel["absorption_250d"] = dl.absorption_ratio(sector_rets, window=250)

fig, ax1 = plt.subplots(figsize=(12, 4))
panel["absorption_250d"].plot(ax=ax1, color="tab:purple", lw=1, label="Absorption ratio (250d)")
panel["avg_corr_60d"].plot(ax=ax1, color="tab:orange", lw=1, alpha=0.8, label="Avg pairwise corr (60d)")
shade_stress(ax1, panel["stress"])
ax1.legend(loc="upper left"); ax1.set_title("Sector co-movement vs. preliminary stress periods")
plt.tight_layout(); plt.show()

In [ ]:
# Quick lead/lag check: correlation between each feature today and STLFSI4 h trading days later.
# Exploratory only; correlation is not a forecast test.
horizons = [0, 5, 20, 60]
rows = {}
for feat in ["avg_corr_60d", "absorption_250d", "VIXCLS", "BAMLH0A0HYM2"]:
    rows[feat] = {f"h={h}": panel[feat].corr(panel["STLFSI4"].shift(-h)) for h in horizons}
pd.DataFrame(rows).T.round(3)

## 6. Save the processed panel

In [ ]:
dl.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out = dl.PROCESSED_DIR / "daily_panel.parquet"
panel.to_parquet(out)
print("Saved", out.relative_to(ROOT), panel.shape)

## Next steps

* **Decide the stress definition** (benchmark index, threshold, episode dating). Context Q1 to Q3.
* **Handle release lags** for weekly indexes before any modeling.
* **NLP data:** pick a text source (news headlines or SEC filings) and plan to score it with the Loughran–McDonald finance word lists.
* **Network component:** Diebold–Yilmaz connectedness on the sector returns.
* **Anomaly detection:** start with an Isolation Forest or rolling z-scores on returns and the features above.